# SPUR Graph Analyst — Live Symbol Explorer

An interactive dashboard over the code-graph index at `.spur/analyst.duckdb`. Two Python source cells publish Arrow ports; a **Deno** artifact cell reads them cross-kernel and renders a `<perspective-viewer>` (Datagrid + d3fc) you can search, filter, group, and pivot in real time.

- **`graph_meta`** — index KPIs (symbols, edges, files, commits)
- **`symbols`** — every crate symbol joined to inbound deps, 90-day churn, and blast-radius score (the searchable substrate)

**User input:** a debounced search box filters the symbol column; `crate` / `kind` dropdowns narrow the set; the Perspective panel adds group-by, aggregation, and sort. Perspective loads its WASM engine from a CDN, so the cell needs active content enabled.

In [2]:
import duckdb
DB = "/Volumes/Projects/spur/.spur/analyst.duckdb"
con = duckdb.connect(DB, read_only=True)
df = con.execute("""
    SELECT node_count, resolved_edge_count, unresolved_edge_count,
           file_count, commit_count, symbol_snapshot_count,
           schema_version, substr(graph_content_hash, 1, 12) AS graph_hash
    FROM _meta
""").df()
con.close()
spur.put("graph_meta", df)
df

node_count,resolved_edge_count,unresolved_edge_count,file_count,commit_count,symbol_snapshot_count,schema_version,graph_hash
51619,83016,94761,2681,2863,579389,spur-graph-schema-v7,4493dcab2d23


,node_count,resolved_edge_count,unresolved_edge_count,file_count,commit_count,symbol_snapshot_count,schema_version,graph_hash
0,51619,83016,94761,2681,2863,579389,spur-graph-schema-v7,4493dcab2d23


In [8]:
import duckdb
DB = "/Volumes/Projects/spur/.spur/analyst.duckdb"
con = duckdb.connect(DB, read_only=True)
# Rich per-symbol analyst table: every crate symbol joined to its inbound
# dependency count, 90-day churn, and blast-radius score. This is the
# searchable substrate the Perspective explorer pivots and filters over.
df = con.execute("""
    SELECT n.qualified_name AS symbol,
           n.symbol_kind   AS kind,
           regexp_replace(n.file_path, '^crates/([^/]+)/.*$', '\\1') AS crate,
           regexp_replace(n.file_path, '^crates/', '') AS file,
           COALESCE(i.inbound_total, 0)::BIGINT AS inbound,
           COALESCE(ch.events, 0)::BIGINT       AS churn_90d,
           round(COALESCE(br.blast_radius_score, 0), 1) AS blast_score,
           COALESCE(strftime(ch.last_touched, '%Y-%m-%d'), '') AS last_touched
    FROM nodes n
    LEFT JOIN v_symbol_inbound   i  USING (stable_symbol_id)
    LEFT JOIN v_symbol_churn_90d ch USING (stable_symbol_id)
    LEFT JOIN v_blast_radius     br USING (stable_symbol_id)
    WHERE n.file_path LIKE 'crates/%'
      AND n.symbol_kind IN ('function', 'method', 'struct', 'enum', 'trait')
    ORDER BY inbound DESC
""").df()
con.close()
spur.put("symbols", df)
print(f"{len(df):,} symbols published to port 'symbols'")
df.head(20)

symbol,kind,crate,file,inbound,churn_90d,blast_score,last_touched
tests::tempdir,function,spur-notebook,spur-notebook/src/recents.rs,508,0,34.7,
impl BrainSessionId::new,method,spur-acp,spur-acp/src/types.rs,183,2,19.9,2026-05-22
SpurEventBody,enum,spur-acp,spur-acp/src/domain/events.rs,155,0,0.0,
impl SessionId::new,method,spur-acp,spur-acp/src/types.rs,144,0,17.7,
impl ExecutorLineage::new,method,spur-core,spur-core/src/lineage/projection.rs,142,0,6.9,


18,507 symbols published to port 'symbols'


,symbol,kind,crate,file,inbound,churn_90d,blast_score,last_touched
0,tests::tempdir,function,spur-notebook,spur-notebook/src/recents.rs,508,0,34.7,
1,impl BrainSessionId::new,method,spur-acp,spur-acp/src/types.rs,183,2,19.9,2026-05-22
2,SpurEventBody,enum,spur-acp,spur-acp/src/domain/events.rs,155,0,0.0,
3,impl SessionId::new,method,spur-acp,spur-acp/src/types.rs,144,0,17.7,
4,impl ExecutorLineage::new,method,spur-core,spur-core/src/lineage/projection.rs,142,0,6.9,
5,impl InputBar::new,method,spur-tui,spur-tui/src/components/input_bar.rs,133,0,16.9,
6,build_facts,function,spur-graph,spur-graph/src/extract/tree_sitter.rs,96,0,8.5,
7,impl McpCallbackServer::new,method,spur-mcp,spur-mcp/src/server/mod.rs,92,0,14.6,
8,write_artifact_parquet,function,spur-graph,spur-graph/src/store/parquet.rs,90,0,12.2,
9,impl SessionSynopsisProjection::new,method,spur-core,spur-core/src/session_synopsis/projection.rs,79,0,3.0,


In [3]:
// open-design artifact -- SPUR Graph Analyst: live Perspective symbol explorer
// Cross-kernel: Python sources publish Arrow ports; this Deno cell reads them
// and hands a columnar payload to a <perspective-viewer> (Datagrid + d3fc).

const meta = await spur.get("graph_meta");
const sy   = await spur.get("symbols");

const g = (name) => {
  const c = meta.getChild(name);
  let x = c.get(0);
  if (typeof x === "bigint") x = Number(x);
  return x;
};
const fmt = (n) => Number(n).toLocaleString("en-US");

// Columnar payload for Perspective (bigint -> number so the engine types it).
const fields = sy.schema.fields.map((f) => f.name);
const DATA = {};
for (const name of fields) {
  const col = sy.getChild(name);
  const out = new Array(sy.numRows);
  for (let i = 0; i < sy.numRows; i++) {
    let x = col.get(i);
    if (typeof x === "bigint") x = Number(x);
    out[i] = x;
  }
  DATA[name] = out;
}
const dataJson = JSON.stringify(DATA);

const searchable = sy.numRows;
const schema = g("schema_version");
const hash   = g("graph_hash");

const kpiData = [
  ["symbols", fmt(g("node_count"))],
  ["resolved edges", fmt(g("resolved_edge_count"))],
  ["unresolved", fmt(g("unresolved_edge_count"))],
  ["files", fmt(g("file_count"))],
  ["commits", fmt(g("commit_count"))],
  ["searchable", fmt(searchable)],
];
const kpiHtml = kpiData
  .map((p) => '<div class="kpi"><div class="v">' + p[1] + '</div><div class="l">' + p[0] + "</div></div>")
  .join("");

const header =
  '<div class="hd"><div><h1>SPUR GRAPH ANALYST</h1>' +
  '<div class="sub">live symbol explorer &middot; search, filter, and pivot the code graph</div></div>' +
  '<div class="fresh">schema <b>' + schema + "</b><br>graph <b>" + hash + "</b><br>" +
  fmt(searchable) + " symbols indexed</div></div>";

const controls =
  '<div class="ctl">' +
  '<input id="q" type="text" placeholder="search symbol... try: reconcile, SpurEvent, ::new, broadcast">' +
  '<label>crate</label><select id="crate"></select>' +
  '<label>kind</label><select id="kind"></select>' +
  '<button id="reset">reset</button></div>';

const foot =
  '<div class="foot"><b>how to use</b> type to filter the symbol column (case-sensitive contains); ' +
  "open the panel on the right to group by crate or kind, aggregate, and re-sort. " +
  "Perspective needs active content enabled. &middot; source: .spur/analyst.duckdb</div>";

const css = `
*{box-sizing:border-box}
html,body{margin:0;height:100%}
body{background:#0a0c0f}
.wrap{font-family:ui-monospace,"SF Mono","JetBrains Mono",Menlo,monospace;
  background:#0a0c0f;color:#e8eaed;padding:18px 20px;min-height:100vh;
  display:flex;flex-direction:column}
.hd{display:flex;align-items:flex-end;justify-content:space-between;
  border-bottom:1px solid #21262e;padding-bottom:12px;margin-bottom:14px}
.hd h1{font-size:16px;letter-spacing:4px;margin:0;font-weight:600}
.hd .sub{color:#8a929e;font-size:11px;letter-spacing:0.6px;margin-top:5px;
  font-family:system-ui,-apple-system,"Segoe UI",sans-serif}
.fresh{color:#8a929e;font-size:10.5px;text-align:right;line-height:1.7}
.fresh b{color:#f2b134;font-weight:600}
.kpis{display:grid;grid-template-columns:repeat(6,1fr);gap:10px;margin-bottom:14px}
.kpi{background:#12151a;border:1px solid #21262e;border-radius:6px;padding:11px 13px}
.kpi .v{font-size:22px;font-weight:600;font-variant-numeric:tabular-nums;line-height:1}
.kpi .l{color:#8a929e;font-size:9px;letter-spacing:1.5px;text-transform:uppercase;
  margin-top:6px}
.ctl{display:flex;gap:9px;align-items:center;margin-bottom:12px;flex-wrap:wrap}
.ctl input,.ctl select{font-family:inherit;font-size:12px;background:#12151a;
  color:#e8eaed;border:1px solid #2a2f37;border-radius:6px;padding:8px 11px;outline:none}
.ctl input{flex:1;min-width:240px}
.ctl input::placeholder{color:#5b6470}
.ctl input:focus,.ctl select:focus{border-color:#f2b134}
.ctl label{color:#646c78;font-size:9px;letter-spacing:1.5px;text-transform:uppercase}
.ctl button{font-family:inherit;font-size:11px;background:#1b2027;color:#cfd3d9;
  border:1px solid #2a2f37;border-radius:6px;padding:8px 13px;cursor:pointer;
  letter-spacing:0.5px}
.ctl button:hover{border-color:#f2b134;color:#f2b134}
perspective-viewer{flex:1;min-height:540px;border:1px solid #21262e;border-radius:6px;
  overflow:hidden}
.foot{color:#646c78;font-size:10px;margin-top:12px;letter-spacing:0.3px;
  font-family:system-ui,-apple-system,"Segoe UI",sans-serif}
.foot b{color:#7e8693;font-weight:500;text-transform:uppercase;letter-spacing:1px;
  margin-right:4px}
`;

const client = [
  "const DATA = " + dataJson + ";",
  'var V = "3.8.0";',
  'const perspective = (await import("https://cdn.jsdelivr.net/npm/@finos/perspective@" + V + "/dist/cdn/perspective.js")).default;',
  'await import("https://cdn.jsdelivr.net/npm/@finos/perspective-viewer@" + V + "/dist/cdn/perspective-viewer.js");',
  'await import("https://cdn.jsdelivr.net/npm/@finos/perspective-viewer-datagrid@" + V + "/dist/cdn/perspective-viewer-datagrid.js");',
  'await import("https://cdn.jsdelivr.net/npm/@finos/perspective-viewer-d3fc@" + V + "/dist/cdn/perspective-viewer-d3fc.js");',
  "const worker = await perspective.worker();",
  "const table = await worker.table(DATA);",
  'const v = document.getElementById("v");',
  "await v.load(table);",
  'await v.restore({plugin:"Datagrid", theme:"Pro Dark", settings:true, sort:[["inbound","desc"]], columns:["symbol","kind","crate","inbound","churn_90d","blast_score","file","last_touched"]});',
  'const q = document.getElementById("q"), cs = document.getElementById("crate"), ks = document.getElementById("kind");',
  'function opts(sel, vals){ sel.innerHTML=""; var a=document.createElement("option"); a.value=""; a.textContent="all"; sel.appendChild(a); vals.forEach(function(x){ var e=document.createElement("option"); e.textContent=x; sel.appendChild(e); }); }',
  "opts(cs, Array.from(new Set(DATA.crate)).sort());",
  "opts(ks, Array.from(new Set(DATA.kind)).sort());",
  'var tmr; function apply(){ var f=[]; var s=q.value.trim(); if(s){ f.push(["symbol","contains",s]); } if(cs.value){ f.push(["crate","==",cs.value]); } if(ks.value){ f.push(["kind","==",ks.value]); } v.restore({filter:f}); }',
  'q.addEventListener("input", function(){ clearTimeout(tmr); tmr=setTimeout(apply,180); });',
  'cs.addEventListener("change", apply);',
  'ks.addEventListener("change", apply);',
  'document.getElementById("reset").addEventListener("click", function(){ q.value=""; cs.value=""; ks.value=""; v.restore({filter:[]}); });',
].join("\n");

const doc =
  '<!doctype html><html><head><meta charset="utf-8">' +
  '<link rel="stylesheet" crossorigin="anonymous" href="https://cdn.jsdelivr.net/npm/@finos/perspective-viewer@3.8.0/dist/css/themes.css">' +
  "<style>" + css + "</style></head><body><div class=\"wrap\">" +
  header +
  '<div class="kpis">' + kpiHtml + "</div>" +
  controls +
  '<perspective-viewer id="v"></perspective-viewer>' +
  foot +
  '<script type="module">' + client + "</script>" +
  "</div></body></html>";

await Deno.jupyter.display({ "text/html": doc }, { raw: true });

<!doctype html> SPUR GRAPH ANALYST live symbol explorer · search, filter, and pivot the code graph schema spur-graph-schema-v7 graph 4493dcab2d23 18,507 symbols indexed 51,619 symbols 83,016 resolved edges 94,761 unresolved 2,681 files 2,863 commits 18,507 searchable crate kind reset how to use type to filter the symbol column (case-sensitive contains); open the panel on the right to group by crate or kind, aggregate, and re-sort. Perspective needs active content enabled. · source: .spur/analyst.duckdb

## Knowledge graph (Cytoscape)

A node-link view of the call + import network. The Perspective grid above is the tabular lens; this is the **topological** one. Two more Arrow ports feed it:

- **`kg_nodes`** — every symbol that participates in a semantic edge (label, qualified name, crate, kind, inbound degree)
- **`kg_edges`** — `calls` / `imports` / `references` / `links` between crate symbols (the structural `contains` tree is excluded)

The full graph is too dense to draw at once, so the search box **seeds an ego-network**: it centers on the highest-impact matching symbol and Cytoscape renders its bounded neighborhood. Click any node to re-center; raise depth to widen the ring.

In [9]:
import duckdb
DB = "/Volumes/Projects/spur/.spur/analyst.duckdb"
con = duckdb.connect(DB, read_only=True)
# Every symbol that participates in a semantic edge (call/import/ref/link),
# with the metadata Cytoscape needs: label, qualified name, crate, kind, inbound.
df = con.execute("""
    WITH kge AS (
        SELECT e.src_id AS a, e.dst_id AS b
        FROM edges e
        JOIN nodes s ON s.node_id = e.src_id
        JOIN nodes d ON d.node_id = e.dst_id
        WHERE e.relation IN ('calls','imports','references','links')
          AND s.file_path LIKE 'crates/%' AND d.file_path LIKE 'crates/%'
    ),
    ids AS (SELECT a AS id FROM kge UNION SELECT b FROM kge)
    SELECT n.node_id::BIGINT AS id,
           n.entity_name      AS label,
           n.qualified_name   AS qn,
           regexp_replace(n.file_path, '^crates/([^/]+)/.*$', '\\1') AS crate,
           n.symbol_kind      AS kind,
           COALESCE(i.inbound_total, 0)::BIGINT AS inbound
    FROM ids
    JOIN nodes n ON n.node_id = ids.id
    LEFT JOIN v_symbol_inbound i ON i.stable_symbol_id = n.stable_symbol_id
""").df()
con.close()
spur.put("kg_nodes", df)
print(f"{len(df):,} graph nodes published to port 'kg_nodes'")
df.head(10)

id,label,qn,crate,kind,inbound
72,apply_cancel_task,apply_cancel_task,spur-mcp,function,2
189,resolve_symbol_for_optional_as_of,resolve_symbol_for_optional_as_of,spur-mcp,function,2
705,build,build,spur-cli,function,1
733,press,press,spur-tui,function,1
737,insert_session,insert_session,spur-cost,function,2


13,542 graph nodes published to port 'kg_nodes'


,id,label,qn,crate,kind,inbound
0,72,apply_cancel_task,apply_cancel_task,spur-mcp,function,2
1,189,resolve_symbol_for_optional_as_of,resolve_symbol_for_optional_as_of,spur-mcp,function,2
2,705,build,build,spur-cli,function,1
3,733,press,press,spur-tui,function,1
4,737,insert_session,insert_session,spur-cost,function,2
5,1019,render_compact_cache_hits_when_generation_stable,render_compact_cache_hits_when_generation_stable,spur-tui,function,1
6,1089,Job,WriteMsg::Job,spur-pm,enum_variant,2
7,1179,seed_registry,seed_registry,spur-tui,function,3
8,1661,expand_stable_symbol_snapshots,expand_stable_symbol_snapshots,spur-graph,function,2
9,1852,modal_rect,modal_rect,spur-tui,function,2


In [10]:
import duckdb
DB = "/Volumes/Projects/spur/.spur/analyst.duckdb"
con = duckdb.connect(DB, read_only=True)
# Semantic edges between crate symbols (the structural 'contains' tree is excluded).
df = con.execute("""
    SELECT e.src_id::BIGINT AS src, e.dst_id::BIGINT AS dst, e.relation AS rel
    FROM edges e
    JOIN nodes s ON s.node_id = e.src_id
    JOIN nodes d ON d.node_id = e.dst_id
    WHERE e.relation IN ('calls','imports','references','links')
      AND s.file_path LIKE 'crates/%' AND d.file_path LIKE 'crates/%'
""").df()
con.close()
spur.put("kg_edges", df)
print(f"{len(df):,} graph edges published to port 'kg_edges'")
df.head(10)

src,dst,rel
11515,4,calls
2053,10,calls
40982,14,calls
35393,47,calls
37146,52,calls


26,988 graph edges published to port 'kg_edges'


,src,dst,rel
0,11515,4,calls
1,2053,10,calls
2,40982,14,calls
3,35393,47,calls
4,37146,52,calls
5,44351,53,calls
6,1767,63,calls
7,13751,65,references
8,6435,72,calls
9,30724,81,calls


In [ ]:
// open-design artifact -- SPUR Knowledge Graph (Cytoscape ego-network)
const N = await spur.get("kg_nodes");
const E = await spur.get("kg_edges");

const idF = N.getChild("id"), labF = N.getChild("label"), qnF = N.getChild("qn"),
      crF = N.getChild("crate"), kiF = N.getChild("kind"), inF = N.getChild("inbound");
const NODES = {};
let maxInb = 1, topId = 0, topN = -1;
const crateSet = new Set();
for (let i = 0; i < N.numRows; i++) {
  const id = Number(idF.get(i));
  const inb = Number(inF.get(i));
  const cr = crF.get(i);
  NODES[id] = { l: labF.get(i), q: qnF.get(i), c: cr, k: kiF.get(i), n: inb };
  if (inb > maxInb) maxInb = inb;
  if (inb > topN) { topN = inb; topId = id; }
  crateSet.add(cr);
}
const sF = E.getChild("src"), dF = E.getChild("dst"), rF = E.getChild("rel");
const EDGES = new Array(E.numRows);
for (let i = 0; i < E.numRows; i++) EDGES[i] = [Number(sF.get(i)), Number(dF.get(i)), rF.get(i)];

const palette = ["#f2b134","#35c2a8","#5aa9e6","#e0719c","#a3be8c","#d08770",
                 "#88c0d0","#ebcb8b","#b48ead","#8fbcbb","#81a1c1","#d6a07a","#bf616a","#9aa7b1"];
const crateList = Array.from(crateSet).sort();
const crateColors = {};
crateList.forEach((c, i) => { crateColors[c] = palette[i % palette.length]; });
const relColors = { calls: "#f2b134", imports: "#5aa9e6", references: "#35c2a8", links: "#e0719c" };

const enc = (s) => s.replace(/</g, "\\u003c");
const nodesJson = enc(JSON.stringify(NODES));
const edgesJson = enc(JSON.stringify(EDGES));
const ccJson = enc(JSON.stringify(crateColors));
const rcJson = enc(JSON.stringify(relColors));

const fmt = (n) => Number(n).toLocaleString("en-US");
const crateLegend = crateList
  .map((c) => '<span class="lg"><i style="background:' + crateColors[c] + '"></i>' + c + "</span>")
  .join("");
const relLegend = Object.keys(relColors)
  .map((r) => '<span class="lg"><i class="ln" style="background:' + relColors[r] + '"></i>' + r + "</span>")
  .join("");

const header =
  '<div class="hd"><div><h1>SPUR KNOWLEDGE GRAPH</h1>' +
  '<div class="sub">call + import network &middot; search seeds an ego-network, click a node to re-center</div></div>' +
  '<div class="fresh">' + fmt(N.numRows) + " symbols<br>" + fmt(E.numRows) +
  " edges<br>calls / imports / refs / links</div></div>";

const controls =
  '<div class="ctl">' +
  '<input id="q" type="text" placeholder="search a symbol to center the graph... try: reconcile, broadcast, SpurEvent">' +
  '<label>depth</label><select id="depth"><option>1</option><option selected>2</option><option>3</option></select>' +
  '<button id="recenter">recenter</button></div>';

const legend =
  '<div class="legend"><span class="ltitle">crates</span>' + crateLegend +
  '<span class="ltitle">edges</span>' + relLegend +
  '<span class="ltitle">node size = inbound deps</span></div>';

const foot =
  '<div class="foot"><b>how to use</b> the search box centers on the highest-impact matching symbol; ' +
  "click any node to re-center on it; raise depth for a wider neighborhood. Bounded to 130 nodes per view. " +
  "Cytoscape loads from a CDN, so active content must be enabled. &middot; source: .spur/analyst.duckdb</div>";

const css = `
*{box-sizing:border-box}
html,body{margin:0;height:100%}
body{background:#0a0c0f}
.wrap{font-family:ui-monospace,"SF Mono","JetBrains Mono",Menlo,monospace;
  background:#0a0c0f;color:#e8eaed;padding:18px 20px;min-height:100vh;
  display:flex;flex-direction:column}
.hd{display:flex;align-items:flex-end;justify-content:space-between;
  border-bottom:1px solid #21262e;padding-bottom:12px;margin-bottom:13px}
.hd h1{font-size:16px;letter-spacing:4px;margin:0;font-weight:600}
.hd .sub{color:#8a929e;font-size:11px;letter-spacing:0.6px;margin-top:5px;
  font-family:system-ui,-apple-system,"Segoe UI",sans-serif}
.fresh{color:#8a929e;font-size:10.5px;text-align:right;line-height:1.7}
.ctl{display:flex;gap:9px;align-items:center;margin-bottom:10px;flex-wrap:wrap}
.ctl input,.ctl select{font-family:inherit;font-size:12px;background:#12151a;
  color:#e8eaed;border:1px solid #2a2f37;border-radius:6px;padding:8px 11px;outline:none}
.ctl input{flex:1;min-width:260px}
.ctl input::placeholder{color:#5b6470}
.ctl input:focus,.ctl select:focus{border-color:#f2b134}
.ctl label{color:#646c78;font-size:9px;letter-spacing:1.5px;text-transform:uppercase}
.ctl button{font-family:inherit;font-size:11px;background:#1b2027;color:#cfd3d9;
  border:1px solid #2a2f37;border-radius:6px;padding:8px 13px;cursor:pointer;letter-spacing:0.5px}
.ctl button:hover{border-color:#f2b134;color:#f2b134}
.status{color:#8a929e;font-size:11px;margin-bottom:8px;min-height:15px;
  font-family:system-ui,-apple-system,"Segoe UI",sans-serif}
.status b{color:#f2b134;font-weight:600}
#cy{flex:1;min-height:520px;background:#0c0f13;border:1px solid #21262e;border-radius:7px}
.legend{display:flex;flex-wrap:wrap;align-items:center;gap:10px 14px;margin-top:11px;
  font-family:system-ui,-apple-system,"Segoe UI",sans-serif;font-size:10.5px;color:#8a929e}
.legend .ltitle{color:#646c78;text-transform:uppercase;letter-spacing:1.2px;font-size:9px}
.lg{display:inline-flex;align-items:center;gap:5px;color:#cfd3d9}
.lg i{width:9px;height:9px;border-radius:2px;display:inline-block}
.lg i.ln{width:14px;height:3px;border-radius:2px}
.foot{color:#646c78;font-size:10px;margin-top:11px;letter-spacing:0.3px;
  font-family:system-ui,-apple-system,"Segoe UI",sans-serif}
.foot b{color:#7e8693;font-weight:500;text-transform:uppercase;letter-spacing:1px;margin-right:4px}
`;

const client = [
  "var NODES = " + nodesJson + ";",
  "var EDGES = " + edgesJson + ";",
  "var CC = " + ccJson + ";",
  "var RC = " + rcJson + ";",
  "var SEED0 = " + topId + ";",
  "var MAXINB = " + maxInb + ";",
  "var adj = {};",
  "function add(a,b,r){ (adj[a]=adj[a]||[]).push({to:b,rel:r}); }",
  "EDGES.forEach(function(e){ add(e[0],e[1],e[2]); add(e[1],e[0],e[2]); });",
  "function nsize(inb){ return 14 + 30*Math.sqrt((inb||0)/MAXINB); }",
  "function ego(seed, depth, cap){",
  "  seed = Number(seed);",
  "  var seen={}; seen[seed]=0; var frontier=[seed]; var order=[seed];",
  "  for(var d=0; d<depth; d++){ var next=[];",
  "    for(var fi=0; fi<frontier.length; fi++){ var id=frontier[fi]; var nb=(adj[id]||[]).slice();",
  "      nb.sort(function(x,y){ return (NODES[y.to]?NODES[y.to].n:0)-(NODES[x.to]?NODES[x.to].n:0); });",
  "      for(var k=0;k<nb.length;k++){ if(order.length>=cap) break; var o=nb[k];",
  "        if(!(o.to in seen) && NODES[o.to]){ seen[o.to]=d+1; order.push(o.to); next.push(o.to); } } }",
  "    frontier=next; if(order.length>=cap) break; }",
  "  var inSet={}; order.forEach(function(x){inSet[x]=1;});",
  "  var els=[];",
  "  order.forEach(function(id){ var m=NODES[id]; els.push({data:{id:String(id), label:m.l, qn:m.q, crate:m.c, kind:m.k, inb:m.n, col:(CC[m.c]||'#888'), sz:nsize(m.n)}}); });",
  "  var es={};",
  "  for(var i=0;i<EDGES.length;i++){ var e=EDGES[i]; if(inSet[e[0]]&&inSet[e[1]]){ var key=e[0]+'-'+e[1]+'-'+e[2]; if(es[key])continue; es[key]=1;",
  "    els.push({data:{id:'x'+key, source:String(e[0]), target:String(e[1]), rel:e[2], ec:(RC[e[2]]||'#555')}}); } }",
  "  return {els:els, n:order.length, e:Object.keys(es).length}; }",
  "var cy = window.cytoscape({ container: document.getElementById('cy'), wheelSensitivity:0.25, style:[",
  "  {selector:'node', style:{ 'background-color':'data(col)', 'width':'data(sz)', 'height':'data(sz)', 'label':'data(label)', 'font-size':9, 'font-family':'ui-monospace,monospace', 'color':'#cfd3d9', 'text-valign':'center', 'text-halign':'right', 'text-margin-x':3, 'min-zoomed-font-size':7 } },",
  "  {selector:'node.seed', style:{ 'border-width':3, 'border-color':'#ffffff', 'color':'#ffffff', 'font-size':12, 'font-weight':'bold', 'z-index':99 } },",
  "  {selector:'edge', style:{ 'width':1, 'line-color':'data(ec)', 'target-arrow-color':'data(ec)', 'target-arrow-shape':'triangle', 'arrow-scale':0.7, 'curve-style':'bezier', 'opacity':0.55 } }",
  "] });",
  "var statusEl = document.getElementById('status');",
  "function render(seed){",
  "  var depth = parseInt(document.getElementById('depth').value,10) || 2;",
  "  var r = ego(seed, depth, 130);",
  "  cy.elements().remove(); cy.add(r.els);",
  "  cy.getElementById(String(Number(seed))).addClass('seed');",
  "  cy.layout({ name:'cose', animate:false, padding:24, nodeRepulsion:function(){return 9000;}, idealEdgeLength:function(){return 70;}, nodeOverlap:8 }).run();",
  "  cy.fit(undefined, 30);",
  "  var m=NODES[Number(seed)];",
  "  statusEl.innerHTML = 'seed <b>' + (m?m.q:seed) + '</b> &middot; ' + r.n + ' nodes, ' + r.e + ' edges (depth ' + depth + ', capped 130)'; }",
  "cy.on('tap','node', function(evt){ render(evt.target.id()); });",
  "function findSeed(q){ q=q.toLowerCase(); var best=null,bn=-1; for(var id in NODES){ var m=NODES[id];",
  "  if((m.q&&m.q.toLowerCase().indexOf(q)>=0)||(m.l&&m.l.toLowerCase().indexOf(q)>=0)){ if(m.n>bn){bn=m.n;best=id;} } } return best; }",
  "var tmr; var qel=document.getElementById('q');",
  "qel.addEventListener('input', function(){ clearTimeout(tmr); tmr=setTimeout(function(){ var s=qel.value.trim(); if(!s) return; var seed=findSeed(s); if(seed!=null){ render(seed); } else { statusEl.innerHTML='no symbol matches: ' + s; } }, 220); });",
  "document.getElementById('depth').addEventListener('change', function(){ render(cy.$('.seed').length?cy.$('.seed').id():SEED0); });",
  "document.getElementById('recenter').addEventListener('click', function(){ qel.value=''; render(SEED0); });",
  "render(SEED0);",
].join("\n");

const doc =
  '<!doctype html><html><head><meta charset="utf-8"><style>' + css + "</style></head>" +
  '<body><div class="wrap">' +
  header + controls + '<div id="status" class="status"></div>' +
  '<div id="cy"></div>' + legend + foot +
  '<script src="https://cdn.jsdelivr.net/npm/cytoscape@3.30.2/dist/cytoscape.min.js"></script>' +
  "<script>" + client + "</script>" +
  "</div></body></html>";

await Deno.jupyter.display({ "text/html": doc }, { raw: true });